# Stochastic variational inference — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Use noisy minibatch natural gradients to scale VI to large data
SVI updates global variational parameters with minibatch estimates. These examples compute a full-data Beta update, a minibatch-scaled update, Robbins-Monro averaging, noise reduction with batch size, and convergence toward full counts.

In [ ]:
# 1) full-data Beta-Bernoulli variational natural parameters
x=np.array([1,0,1,1,0,1,1,1,0,1]); a0,b0=1.,1.; full=np.array([a0+x.sum(), b0+len(x)-x.sum()])
plt.figure(figsize=(6,3)); plt.bar(['alpha','beta'],full); plt.title('full-data coordinate update')
assert np.allclose(full,[8,4])

In [ ]:
# 2) minibatch update scales sufficient statistics by N/B
mb=x[:4]; N=len(x); B=len(mb); noisy=np.array([a0+N/B*mb.sum(), b0+N/B*(B-mb.sum())])
plt.figure(figsize=(6,3)); plt.bar(['alpha','beta'],noisy); plt.title('minibatch-scaled natural parameter')
assert np.allclose(noisy,[8.5,3.5])

In [ ]:
# 3) Robbins-Monro step blends old and noisy natural parameters
lam=np.array([1.,1.]); rho=0.3; upd=(1-rho)*lam+rho*noisy
plt.figure(figsize=(6,3)); plt.bar(['old alpha','new alpha'],[lam[0],upd[0]]); plt.title('partial natural-gradient step')
assert np.allclose(upd,[3.25,1.75])

In [ ]:
# 4) larger minibatches reduce estimator variance
rng=np.random.default_rng(2); data=rng.binomial(1,0.7,1000); vars=[]
for B in [5,20,100]:
    est=[1000/B*rng.choice(data,B,replace=False).sum() for _ in range(200)]; vars.append(np.var(est))
plt.figure(figsize=(6,3)); plt.plot([5,20,100],vars,'-o'); plt.xlabel('batch size'); plt.ylabel('variance'); plt.title('larger batches make quieter updates')
assert vars[0]>vars[-1]

In [ ]:
# 5) repeated SVI moves toward the full update
rng=np.random.default_rng(3); lam=np.array([1.,1.]); path=[]
for t in range(1,80):
    mb=rng.choice(x,4,replace=False); noisy=np.array([a0+N/4*mb.sum(), b0+N/4*(4-mb.sum())]); rho=(t+5)**-0.6; lam=(1-rho)*lam+rho*noisy; path.append(lam.copy())
plt.figure(figsize=(6,3)); plt.plot(np.array(path)); plt.title('SVI path')
assert np.linalg.norm(lam-full)<1.5